In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

DATA_PATH = "/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/ml_service/data/"
MODEL_PATH = "/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/ml_service/models/"

# Veriyi yukle
df = pd.read_csv(DATA_PATH + "islenmis_veri.csv")
df["tarih"] = pd.to_datetime(df["tarih"])
df = df.sort_values(["sehir", "tarih"]).reset_index(drop=True)

print("=" * 40)
print("VERI GENEL BILGI")
print("=" * 40)
print(f"Toplam satir: {len(df)}")
print(f"Tarih araligi: {df['tarih'].min()} → {df['tarih'].max()}")
print(f"\nIlk 5 satir:")
print(df.head())

VERI GENEL BILGI
Toplam satir: 21920
Tarih araligi: 2022-01-01 00:00:00 → 2024-12-31 00:00:00

Ilk 5 satir:
       tarih  max_sicaklik  min_sicaklik  yagis  ruzgar_hizi  sehir    bolge  \
0 2022-01-01          17.9           9.1    0.0         13.1  Adana  Akdeniz   
1 2022-01-02          16.5           6.8    0.0         18.1  Adana  Akdeniz   
2 2022-01-03          15.2           5.8    0.0         17.6  Adana  Akdeniz   
3 2022-01-04          15.9           7.2    0.0         12.0  Adana  Akdeniz   
4 2022-01-05          17.3           6.1    0.0         11.0  Adana  Akdeniz   

  etiket  
0  uygun  
1  uygun  
2  uygun  
3  uygun  
4  uygun  


In [2]:
# LSTM icin sadece Bitlis verisini kullanalim
# (ornek sehir olarak)
bitlis_df = df[df["sehir"] == "Bitlis"].copy()
bitlis_df = bitlis_df.reset_index(drop=True)

print(f"Bitlis veri sayisi: {len(bitlis_df)}")

# Sadece sayisal sutunlari al
ozellikler = ["max_sicaklik", "min_sicaklik", "yagis", "ruzgar_hizi"]
veri = bitlis_df[ozellikler].values

# MinMaxScaler ile normalize et (0-1 arasi)
lstm_scaler = MinMaxScaler()
veri_normalize = lstm_scaler.fit_transform(veri)

# Zaman serisi penceresi olustur
# Son 30 gune bakarak sonraki gunu tahmin edecek
PENCERE = 30

X_lstm, y_lstm = [], []
for i in range(PENCERE, len(veri_normalize)):
    X_lstm.append(veri_normalize[i-PENCERE:i])
    y_lstm.append(veri_normalize[i, 0])  # max_sicaklik tahmin edecek

X_lstm = np.array(X_lstm)
y_lstm = np.array(y_lstm)

print(f"X sekli: {X_lstm.shape}")
print(f"y sekli: {y_lstm.shape}")

# Egitim ve test olarak bol
bolme = int(len(X_lstm) * 0.8)
X_train = X_lstm[:bolme]
X_test  = X_lstm[bolme:]
y_train = y_lstm[:bolme]
y_test  = y_lstm[bolme:]

print(f"\nEgitim: {len(X_train)} ornek")
print(f"Test  : {len(X_test)} ornek")

Bitlis veri sayisi: 1096
X sekli: (1066, 30, 4)
y sekli: (1066,)

Egitim: 852 ornek
Test  : 214 ornek


In [3]:
# LSTM modeli olustur
model_lstm = Sequential([
    LSTM(64, return_sequences=True, 
         input_shape=(PENCERE, len(ozellikler))),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1)
])

model_lstm.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model_lstm.summary()

# Erken durdurma - model iyilesmeyi birakirsa dur
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

print("\nModel egitimi basliyor...")

# Modeli egit
tarihce = model_lstm.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

print("\nEgitim tamamlandi!")

/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/.venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,625 (119.63 KB)

 Trainable params: 30,625 (119.63 KB)

 Non-trainable params: 0 (0.00 B)


Model egitimi basliyor...
Epoch 1/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 0.0825 - mae: 0.2106 - val_loss: 0.0125 - val_mae: 0.0944
Epoch 2/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0179 - mae: 0.1054 - val_loss: 0.0094 - val_mae: 0.0804
Epoch 3/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0116 - mae: 0.0849 - val_loss: 0.0131 - val_mae: 0.0952
Epoch 4/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0105 - mae: 0.0818 - val_loss: 0.0106 - val_mae: 0.0838
Epoch 5/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0102 - mae: 0.0800 - val_loss: 0.0141 - val_mae: 0.0995
Epoch 6/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0089 - mae: 0.0738 - val_loss: 0.0102 - val_mae: 0.0819
Epoch 7/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0086 - mae: 0.0726 - val_loss: 0.0116 - val_mae: 0.0889
Epoch 8/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0087 - mae: 0.0728 - val_loss: 0.0112 - val_mae: 0.0861
Epoch 9/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8m

In [4]:
# Test seti uzerinde tahmin yap
y_pred = model_lstm.predict(X_test)

# Gercek ve tahmin degerlerini geri donustur
# (normalize edilmis degerlerden gercek degerler)
y_test_gercek = lstm_scaler.inverse_transform(
    np.concatenate([y_test.reshape(-1,1), 
                    np.zeros((len(y_test), 3))], axis=1)
)[:, 0]

y_pred_gercek = lstm_scaler.inverse_transform(
    np.concatenate([y_pred, 
                    np.zeros((len(y_pred), 3))], axis=1)
)[:, 0]

# Hata metrikleri
mae = np.mean(np.abs(y_test_gercek - y_pred_gercek))
rmse = np.sqrt(np.mean((y_test_gercek - y_pred_gercek)**2))

print("=" * 40)
print("LSTM MODEL PERFORMANSI")
print("=" * 40)
print(f"MAE  (Ortalama Mutlak Hata) : {mae:.2f} °C")
print(f"RMSE (Kok Ortalama Kare Hata): {rmse:.2f} °C")

print("\nGercek vs Tahmin (ilk 10 gun):")
karsilastirma = pd.DataFrame({
    "Gercek": y_test_gercek[:10].round(1),
    "Tahmin": y_pred_gercek[:10].round(1),
    "Fark"  : np.abs(y_test_gercek[:10] - 
                     y_pred_gercek[:10]).round(1)
})
print(karsilastirma)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
LSTM MODEL PERFORMANSI
MAE  (Ortalama Mutlak Hata) : 2.90 °C
RMSE (Kok Ortalama Kare Hata): 3.96 °C

Gercek vs Tahmin (ilk 10 gun):
   Gercek  Tahmin  Fark
0    21.7    19.7   2.0
1    23.2    20.2   3.0
2    24.9    20.6   4.3
3    25.4    21.0   4.4
4    25.6    21.4   4.2
5    27.0    21.9   5.1
6    28.1    22.3   5.8
7    26.8    22.8   4.0
8    25.4    23.3   2.1
9    25.7    23.7   2.0


In [5]:
# Gelecek 7 gun tahmini
son_30_gun = veri_normalize[-PENCERE:]
tahminler = []

for _ in range(7):
    girdi = son_30_gun[-PENCERE:].reshape(1, PENCERE, len(ozellikler))
    tahmin = model_lstm.predict(girdi, verbose=0)[0][0]
    
    # Yeni satir olustur
    yeni_satir = son_30_gun[-1].copy()
    yeni_satir[0] = tahmin
    son_30_gun = np.vstack([son_30_gun, yeni_satir])
    tahminler.append(tahmin)

# Gercek degerler
tahminler_gercek = lstm_scaler.inverse_transform(
    np.concatenate([
        np.array(tahminler).reshape(-1,1),
        np.zeros((7, 3))
    ], axis=1)
)[:, 0]

print("=" * 40)
print("GELECEK 7 GUN MAX SICAKLIK TAHMINI (Bitlis)")
print("=" * 40)
for i, sicaklik in enumerate(tahminler_gercek):
    print(f"  Gun {i+1}: {sicaklik:.1f} °C")

# Modeli kaydet
model_lstm.save(MODEL_PATH + "lstm_model.keras")
joblib.dump(lstm_scaler, MODEL_PATH + "lstm_scaler.pkl")

print("\n" + "=" * 40)
print("Model kaydedildi!")
print("  → models/lstm_model.keras")
print("  → models/lstm_scaler.pkl")

GELECEK 7 GUN MAX SICAKLIK TAHMINI (Bitlis)
  Gun 1: 5.2 °C
  Gun 2: 5.0 °C
  Gun 3: 4.9 °C
  Gun 4: 4.8 °C
  Gun 5: 4.7 °C
  Gun 6: 4.6 °C
  Gun 7: 4.5 °C

Model kaydedildi!
  → models/lstm_model.keras
  → models/lstm_scaler.pkl
